# MobileNet Transfer Learning — Calving Event Classification at Perito Moreno Glacier

**Associated paper:**  
> *[Paper title — to be filled]*  
> Authors: [Author list]  
> Journal: [Journal name]  
> DOI: [DOI]

**Description:**  
This notebook fine-tunes a MobileNet backbone (pre-trained on ImageNet, frozen weights)
for binary classification of calving events at Perito Moreno Glacier from time-lapse imagery.
A small fully-connected head is trained on top of the frozen feature extractor.
Input shape: `(256, 256, 3)`.

**Dataset:**  
Pre-split `.npy` arrays (train / validation / test) must be placed in `DATA_DIR`.
The dataset was partitioned 80 % train / 10 % validation / 10 % test.
See `README.md` for the expected directory structure.

**Requirements:** `tensorflow ≥ 2.10`, `scikit-learn`, `matplotlib`, `seaborn`, `numpy`

## 0 · Configuration

In [ ]:
import os

# ── Paths (edit these to match your local layout) ────────────────────────────
DATA_DIR   = "data"      # folder with train/val/test .npy files
OUTPUT_DIR = "outputs"   # folder for saved model, predictions, history

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1 · Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import confusion_matrix, f1_score

print("TensorFlow version:", tf.__version__)

## 2 · Load Dataset

In [ ]:
train_images = np.load(os.path.join(DATA_DIR, "train_images.npy"))
train_labels = np.load(os.path.join(DATA_DIR, "train_labels.npy"))
val_images   = np.load(os.path.join(DATA_DIR, "val_images.npy"))
val_labels   = np.load(os.path.join(DATA_DIR, "val_labels.npy"))
test_images  = np.load(os.path.join(DATA_DIR, "test_images.npy"))
test_labels  = np.load(os.path.join(DATA_DIR, "test_labels.npy"))

print("train_images:", train_images.shape, "| train_labels:", train_labels.shape)
print("val_images  :", val_images.shape,   "| val_labels  :", val_labels.shape)
print("test_images :", test_images.shape,  "| test_labels :", test_labels.shape)

## 3 · Model Architecture

MobileNet backbone (ImageNet weights, all layers frozen) followed by a small
fully-connected classification head.  
Freezing the convolutional base allows training with limited labelled data while
retaining ImageNet-learned feature representations.

In [ ]:
input_shape = (256, 256, 3)

# Load MobileNet as frozen feature extractor
base_model = MobileNet(weights="imagenet", include_top=False, input_shape=input_shape)
base_model.trainable = False  # freeze all convolutional weights

model = keras.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(8,  activation="relu"),
    layers.Dense(1,  activation="sigmoid"),
])

model.summary()

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

## 4 · Training

In [ ]:
history = model.fit(
    train_images, train_labels,
    epochs=200,
    batch_size=40,
    validation_data=(val_images, val_labels),
    verbose=2,
)

In [ ]:
np.save(os.path.join(OUTPUT_DIR, "history_loss_mnet.npy"),         history.history["loss"])
np.save(os.path.join(OUTPUT_DIR, "history_val_loss_mnet.npy"),     history.history["val_loss"])
np.save(os.path.join(OUTPUT_DIR, "history_accuracy_mnet.npy"),     history.history["accuracy"])
np.save(os.path.join(OUTPUT_DIR, "history_val_accuracy_mnet.npy"), history.history["val_accuracy"])

## 5 · Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["loss"],     label="Training Loss",   color="k")
axes[0].plot(history.history["val_loss"], label="Validation Loss", color="gray")
axes[0].set_yscale("log")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training and Validation Loss")
axes[0].legend()
axes[0].grid(color="g", linestyle="--", linewidth=0.5)

axes[1].plot(history.history["accuracy"],     label="Training Accuracy",   color="k")
axes[1].plot(history.history["val_accuracy"], label="Validation Accuracy", color="gray")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Training and Validation Accuracy")
axes[1].legend()
axes[1].grid(color="g", linestyle="--", linewidth=0.5)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves_mnet.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6 · Model Evaluation

In [ ]:
train_loss, train_acc = model.evaluate(train_images, train_labels, verbose=0)
val_loss,   val_acc   = model.evaluate(val_images,   val_labels,   verbose=0)
test_loss,  test_acc  = model.evaluate(test_images,  test_labels,  verbose=0)

print(f"Train      — loss: {train_loss:.4f} | accuracy: {train_acc:.4f}")
print(f"Validation — loss: {val_loss:.4f}   | accuracy: {val_acc:.4f}")
print(f"Test       — loss: {test_loss:.4f}  | accuracy: {test_acc:.4f}")

### 6.1 · Optimal Decision Threshold (F1-based)

The default sigmoid threshold of 0.5 may not be optimal for imbalanced datasets.
Here we sweep thresholds on the **validation set** and select the one that maximises F1-score.
This threshold is then applied consistently to all evaluation splits.

In [ ]:
val_prob = model.predict(val_images, verbose=0).ravel()
ths  = np.linspace(0, 1, 501)
f1s  = [f1_score(val_labels, (val_prob >= t).astype(int), zero_division=0) for t in ths]
t_opt = ths[np.argmax(f1s)]
print(f"Optimal F1 threshold (validation set): {t_opt:.3f}  |  max F1 = {max(f1s):.4f}")

In [ ]:
y_pred_val   = (model.predict(val_images,   verbose=0).ravel() >= t_opt).astype(int)
y_pred_test  = (model.predict(test_images,  verbose=0).ravel() >= t_opt).astype(int)
y_pred_train = (model.predict(train_images, verbose=0).ravel() >= t_opt).astype(int)

np.save(os.path.join(OUTPUT_DIR, "y_pred_val_mnet.npy"),   y_pred_val)
np.save(os.path.join(OUTPUT_DIR, "y_pred_test_mnet.npy"),  y_pred_test)
np.save(os.path.join(OUTPUT_DIR, "y_pred_train_mnet.npy"), y_pred_train)

### 6.2 · Confusion Matrices

In [ ]:
def plot_confusion(y_true, y_pred, title, ax):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=["No Calving", "Calving"],
                yticklabels=["No Calving", "Calving"], ax=ax)
    ax.set_title(title)
    ax.set_ylabel("True Label")
    ax.set_xlabel("Predicted Label")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_confusion(val_labels,  y_pred_val,  "Confusion Matrix — Validation", axes[0])
plot_confusion(test_labels, y_pred_test, "Confusion Matrix — Test",       axes[1])
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrices_mnet.png"), dpi=150, bbox_inches="tight")
plt.show()

### 6.3 · Sample Predictions (Validation Set)

In [ ]:
num_images = 10
fig, axes = plt.subplots(2, num_images, figsize=(20, 4))

for i in range(num_images):
    img_channel = val_images[i].reshape(256, 256, 3)[:, :, 0]
    axes[0, i].imshow(img_channel, cmap="gray")
    axes[0, i].set_title(f"True: {int(val_labels[i])}")
    axes[0, i].axis("off")

    axes[1, i].imshow(img_channel, cmap="gray")
    axes[1, i].set_title(f"Pred: {y_pred_val[i]}")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Ground Truth", fontsize=10)
axes[1, 0].set_ylabel("Prediction",   fontsize=10)
plt.suptitle("Sample Validation Predictions (0 = No Calving, 1 = Calving)", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "sample_predictions_mnet.png"), dpi=150, bbox_inches="tight")
plt.show()

## 7 · Save Model

In [ ]:
model_path = os.path.join(OUTPUT_DIR, "MNet_calving.keras")
model.save(model_path)
print(f"Model saved to: {model_path}")

# Verify the saved model loads correctly
model_calving = load_model(model_path)
print("Model loaded successfully. Input shape:", model_calving.input_shape)